# GossipRoboFL — Experiment Analysis

This notebook loads saved experiment results from `results/` and generates all publication-quality plots.

**Before running:** complete at least one experiment run:
```bash
python main.py                                    # gossip baseline
python main.py --mode fedavg                      # FedAvg baseline  
python experiments/ablation_byzantine.py          # robustness sweep
python experiments/ablation_topology.py           # topology sweep
python experiments/ablation_heterogeneity.py      # heterogeneity sweep (non-IID + straggler)
```

In [ ]:
import os
import sys
import json
import glob

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Add project root to path
sys.path.insert(0, os.path.dirname(os.getcwd()))

from src.metrics import (
    load_history,
    plot_convergence_curves,
    plot_accuracy_vs_byzantine_fraction,
    plot_communication_overhead,
    plot_non_iid_impact,
    generate_topology_gif,
)

RESULTS_DIR = '../results'
os.makedirs(RESULTS_DIR, exist_ok=True)

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'legend.fontsize': 9,
    'lines.linewidth': 2,
})

print('Setup complete.')

## 1. Load All Experiment Logs

In [ ]:
def load_all_histories(results_dir: str) -> dict[str, list[dict]]:
    """Load all *_metrics.json files from results directory."""
    histories = {}
    for path in sorted(glob.glob(os.path.join(results_dir, '*_metrics.json'))):
        name = os.path.basename(path).replace('_metrics.json', '')
        try:
            with open(path) as f:
                histories[name] = json.load(f)
            n_rounds = len(histories[name])
            has_acc = any('honest_global_accuracy' in e for e in histories[name])
            print(f'  {name}: {n_rounds} rounds, accuracy logged={has_acc}')
        except Exception as e:
            print(f'  [skip] {name}: {e}')
    return histories

print(f'Scanning {RESULTS_DIR}...')
histories = load_all_histories(RESULTS_DIR)
print(f'\nLoaded {len(histories)} experiment(s).')

## 2. Convergence Curves — Gossip vs FedAvg

In [ ]:
# Filter for baseline comparisons (gossip mean/ssclip vs fedavg)
# Exclude heterogeneity ablation runs (noniid_*) from baseline comparison
baseline_keys = [
    k for k in histories
    if any(tag in k for tag in ['fedavg', 'aggmean', 'aggssclip', 'default'])
    and not k.startswith('noniid_')
]

if baseline_keys:
    baseline_histories = {k: histories[k] for k in baseline_keys}
    fig = plot_convergence_curves(
        baseline_histories,
        metric='honest_global_accuracy',
        title='Gossip FL vs Centralised FedAvg',
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_convergence_comparison.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No baseline experiments found. Run main.py first.')

## 3. Byzantine Robustness

In [ ]:
byz_json = os.path.join(RESULTS_DIR, 'ablation_byzantine_results.json')

if os.path.exists(byz_json):
    with open(byz_json) as f:
        byz_raw = json.load(f)
    # Convert string keys back to float
    byz_results = {float(k): v for k, v in byz_raw.items()}

    fig = plot_accuracy_vs_byzantine_fraction(
        byz_results,
        title='Byzantine Robustness: SSClip vs ClippedGossip vs Mean (sign_flip attack)',
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_byzantine_robustness.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Print table
    methods = list(next(iter(byz_results.values())).keys())
    print(f"{'f':>6} | " + ' | '.join(f'{m:>15}' for m in methods))
    print('-' * (10 + 18 * len(methods)))
    for f, accs in sorted(byz_results.items()):
        row = f'{f:>6.1f} | ' + ' | '.join(f"{accs.get(m, float('nan')):>15.4f}" for m in methods)
        print(row)
else:
    print(f'Not found: {byz_json}. Run: python experiments/ablation_byzantine.py')

## 4. Communication Overhead

In [ ]:
# Pick a non-fedavg, non-ablation run to show comm overhead
gossip_keys = [
    k for k in histories
    if 'fedavg' not in k and not k.startswith('noniid_') and histories[k]
]

if gossip_keys:
    key = gossip_keys[0]
    print(f'Showing communication overhead for: {key}')
    fig = plot_communication_overhead(
        histories[key],
    )
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_comm_overhead.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No gossip experiment found. Run main.py first.')

## 5. Loss Curves

In [ ]:
# Plot global_loss over rounds for baseline runs
loss_keys = [
    k for k in histories
    if not k.startswith('noniid_')
    and any('global_loss' in e for e in histories[k])
]

if loss_keys:
    fig, ax = plt.subplots(figsize=(9, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(loss_keys), 1)))
    for (key, color) in zip(loss_keys[:8], colors):
        hist = histories[key]
        rounds = [e['round'] for e in hist if 'global_loss' in e]
        losses = [e['global_loss'] for e in hist if 'global_loss' in e]
        ax.plot(rounds, losses, label=key[:30], color=color)
    ax.set_xlabel('Round')
    ax.set_ylabel('Mean Test Loss')
    ax.set_title('Test Loss Over Rounds')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_loss_curves.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No global_loss data found. Run main.py first.')

## 6. Non-IID Impact (Dirichlet Alpha)

Results come from `ablation_heterogeneity.py` which saves:
- Per-run `noniid_a{alpha}_hetFalse_metrics.json` files
- Summary in `ablation_heterogeneity_results.json`

In [ ]:
ALPHA_VALUES = [0.1, 0.5, 1.0, 10.0]

# Load convergence histories for homogeneous (hetFalse) runs only
# Naming convention from ablation_heterogeneity.py: noniid_a{alpha}_het{False|True}
alpha_histories: dict[float, list[dict]] = {}
for alpha in ALPHA_VALUES:
    for key, hist in histories.items():
        # Match noniid_a{alpha}_hetFalse specifically to avoid mixing in straggler runs
        if f'noniid_a{alpha}_hetFalse' in key or f'noniid_a{alpha}_het0' in key:
            alpha_histories[alpha] = hist
            break
    else:
        # Fallback: any key containing _a{alpha}_ that isn't a het=True run
        for key, hist in histories.items():
            if f'_a{alpha}' in key and 'hetTrue' not in key and 'het1' not in key:
                alpha_histories[alpha] = hist
                break

# Also load summary bar chart data from consolidated JSON if available
het_json = os.path.join(RESULTS_DIR, 'ablation_heterogeneity_results.json')
alpha_summary = None
if os.path.exists(het_json):
    with open(het_json) as f:
        het_results = json.load(f)
    alpha_summary = het_results.get('alpha_ablation')  # list of {alpha, final_accuracy}

print(f'Alpha histories loaded: {sorted(alpha_histories.keys())}')
if alpha_summary:
    print(f'Summary JSON: {len(alpha_summary)} alpha entries')
else:
    print('No summary JSON found (run ablation_heterogeneity.py for bar chart data)')

# --- Convergence curves ---
if alpha_histories:
    fig = plot_non_iid_impact(
        alpha_histories,
        save_path=f'{RESULTS_DIR}/nb_noniid_convergence.png',
    )
    plt.show()
    plt.close(fig)
else:
    print('No alpha history files found. Run: python experiments/ablation_heterogeneity.py')

# --- Final accuracy bar chart (from consolidated JSON or extracted from histories) ---
if not alpha_summary and alpha_histories:
    # Build summary from loaded histories
    alpha_summary = []
    for alpha, hist in sorted(alpha_histories.items()):
        acc_entries = [e for e in hist if 'honest_global_accuracy' in e]
        final_acc = acc_entries[-1]['honest_global_accuracy'] if acc_entries else float('nan')
        alpha_summary.append({'alpha': alpha, 'final_accuracy': final_acc})

if alpha_summary:
    alphas = [s['alpha'] for s in alpha_summary]
    accs = [s['final_accuracy'] for s in alpha_summary]
    fig2, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar([str(a) for a in alphas], accs, color='#3498db', edgecolor='white', width=0.5)
    ax.bar_label(bars, fmt='%.3f', fontsize=10, padding=3)
    ax.set_xlabel('Dirichlet α (higher = more IID)', fontsize=12)
    ax.set_ylabel('Final Test Accuracy', fontsize=12)
    ax.set_title('Final Accuracy vs Non-IID Degree', fontsize=13)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig2.savefig(f'{RESULTS_DIR}/nb_noniid_bar.png', dpi=150, bbox_inches='tight')
    plt.close(fig2)

## 7. Client Heterogeneity (Homogeneous vs Straggler Robots)

Compares homogeneous local epochs vs heterogeneous (straggler) robots.
Data from `ablation_heterogeneity.py`: runs named `noniid_a0.5_hetFalse` and `noniid_a0.5_hetTrue`.

In [ ]:
# Load heterogeneity comparison histories
het_histories: dict[str, list[dict]] = {}
for key, hist in histories.items():
    if 'noniid_a0.5_hetFalse' in key or 'noniid_a0.5_het0' in key:
        het_histories['homogeneous'] = hist
    elif 'noniid_a0.5_hetTrue' in key or 'noniid_a0.5_het1' in key:
        het_histories['heterogeneous (stragglers)'] = hist

# Final accuracy comparison from consolidated JSON
het_final: dict[str, float] | None = None
if os.path.exists(het_json):
    with open(het_json) as f:
        het_data = json.load(f)
    het_final = het_data.get('heterogeneity_ablation')  # {homogeneous: acc, heterogeneous: acc}

if het_histories:
    fig = plot_convergence_curves(
        het_histories,
        metric='honest_global_accuracy',
        title='Client Heterogeneity Impact (α=0.5)',
        save_path=f'{RESULTS_DIR}/nb_heterogeneity_comparison.png',
    )
    plt.show()
    plt.close(fig)
elif het_final:
    # Only summary available — show bar chart
    fig, ax = plt.subplots(figsize=(6, 4))
    labels = list(het_final.keys())
    vals = [het_final[l] for l in labels]
    bars = ax.bar(labels, vals, color=['#3498db', '#e74c3c'], edgecolor='white', width=0.4)
    ax.bar_label(bars, fmt='%.3f', fontsize=11, padding=3)
    ax.set_ylabel('Final Test Accuracy')
    ax.set_title('Client Heterogeneity: Final Accuracy (α=0.5)')
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_heterogeneity_bar.png', dpi=150, bbox_inches='tight')
    plt.close(fig)
else:
    print('No heterogeneity data found. Run: python experiments/ablation_heterogeneity.py')

## 8. Topology Evolution GIF

In [ ]:
import glob
from IPython.display import Image as IPImage, display

# Check for existing GIF
gif_files = glob.glob(os.path.join(RESULTS_DIR, '*_topology.gif'))
if gif_files:
    print(f'Showing: {gif_files[0]}')
    display(IPImage(filename=gif_files[0]))
else:
    # Try to generate from snapshots
    snap_dir = os.path.join(RESULTS_DIR, 'topology_snaps')
    snaps = sorted(glob.glob(os.path.join(snap_dir, '*.png')))
    if snaps:
        gif_path = os.path.join(RESULTS_DIR, 'topology_evolution.gif')
        generate_topology_gif(snaps, gif_path, fps=3)
        display(IPImage(filename=gif_path))
    else:
        print(f'No topology snapshots found in {snap_dir}.')
        print('Run main.py to generate topology snapshots (saved every update_every rounds).')

## 9. Topology Density Ablation

In [ ]:
topo_json = os.path.join(RESULTS_DIR, 'ablation_topology_results.json')

if os.path.exists(topo_json):
    with open(topo_json) as f:
        topo_results = json.load(f)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    rs = [s['comm_range'] for s in topo_results]
    final_accs = [s['final_accuracy'] for s in topo_results]
    rtt = [s['rounds_to_target'] for s in topo_results]
    gaps = [s['mean_spectral_gap'] for s in topo_results]

    axes[0].plot(rs, final_accs, marker='o', color='#3498db')
    axes[0].set_xlabel('Comm. Range r')
    axes[0].set_ylabel('Final Accuracy')
    axes[0].set_title('Accuracy vs Topology Density')
    axes[0].grid(alpha=0.3)

    axes[1].plot(rs, rtt, marker='s', color='#e74c3c')
    axes[1].set_xlabel('Comm. Range r')
    axes[1].set_ylabel('Rounds to 60% Accuracy')
    axes[1].set_title('Convergence Speed vs Density')
    axes[1].grid(alpha=0.3)

    axes[2].plot(rs, gaps, marker='^', color='#2ecc71')
    axes[2].set_xlabel('Comm. Range r')
    axes[2].set_ylabel('Mean Spectral Gap')
    axes[2].set_title('Algebraic Connectivity vs Density')
    axes[2].grid(alpha=0.3)

    plt.suptitle('Topology Density Ablation (SSClip, no attack)', fontsize=13)
    plt.tight_layout()
    plt.show()
    fig.savefig(f'{RESULTS_DIR}/nb_topology_ablation.png', dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Table
    print(f"{'r':>6} | {'Final Acc':>10} | {'Rounds->60%':>12} | {'Spectral Gap':>12}")
    print('-' * 50)
    for s in topo_results:
        print(f"{s['comm_range']:>6.1f} | {s['final_accuracy']:>10.4f} | {s['rounds_to_target']:>12d} | {s['mean_spectral_gap']:>12.4f}")
else:
    print(f'Not found: {topo_json}. Run: python experiments/ablation_topology.py')

## 10. Summary Dashboard

2×2 overview: convergence, Byzantine robustness, communication overhead, Non-IID impact.

In [ ]:
# Combined 2x2 dashboard of key results
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])  # Convergence
ax2 = fig.add_subplot(gs[0, 1])  # Byzantine robustness
ax3 = fig.add_subplot(gs[1, 0])  # Comm overhead
ax4 = fig.add_subplot(gs[1, 1])  # Non-IID

colors = plt.cm.tab10(np.linspace(0, 1, 5))

# Panel 1: Convergence curves (baseline runs only)
plotted = False
baseline_display = [
    (name, hist) for name, hist in list(histories.items())[:5]
    if not name.startswith('noniid_')
]
for i, (name, hist) in enumerate(baseline_display):
    rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
    accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
    if rounds:
        ax1.plot(rounds, accs, label=name[:25], color=colors[i % 5])
        plotted = True
ax1.set_xlabel('Round')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('Convergence Curves')
ax1.set_ylim(0, 1.05)
ax1.legend(fontsize=7)
ax1.grid(alpha=0.3)
if not plotted:
    ax1.text(0.5, 0.5, 'No data — run experiments first', ha='center', va='center', transform=ax1.transAxes)

# Panel 2: Byzantine robustness
if os.path.exists(byz_json):
    with open(byz_json) as f:
        byz_raw = json.load(f)
    byz_results = {float(k): v for k, v in byz_raw.items()}
    fractions = sorted(byz_results.keys())
    methods = list(next(iter(byz_results.values())).keys())
    for i, method in enumerate(methods):
        accs = [byz_results[f].get(method, np.nan) for f in fractions]
        ax2.plot([f*100 for f in fractions], accs, marker='o', label=method, color=colors[i])
    ax2.set_xlabel('Byzantine Fraction f (%)')
    ax2.set_ylabel('Final Accuracy')
    ax2.set_title('Byzantine Robustness')
    ax2.set_ylim(0, 1.05)
    ax2.legend()
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'Run ablation_byzantine.py', ha='center', va='center', transform=ax2.transAxes)
    ax2.set_title('Byzantine Robustness')

# Panel 3: Comm overhead
gossip_display = [
    k for k in histories
    if 'fedavg' not in k and not k.startswith('noniid_') and histories[k]
]
if gossip_display:
    hist = histories[gossip_display[0]]
    rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
    accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
    comm = [e.get('cumulative_comm_bytes', 0) / 1e6 for e in hist if 'honest_global_accuracy' in e]
    ax3.plot(rounds, accs, color='#3498db', label='Accuracy')
    ax3b = ax3.twinx()
    ax3b.plot(rounds, comm, color='#e74c3c', linestyle='--', label='Comm (MB)')
    ax3.set_xlabel('Round')
    ax3.set_ylabel('Accuracy', color='#3498db')
    ax3b.set_ylabel('Cumul. Comm (MB)', color='#e74c3c')
    ax3.set_title('Accuracy vs Communication')
    ax3.set_ylim(0, 1.05)
    ax3.grid(alpha=0.3)
else:
    ax3.text(0.5, 0.5, 'Run main.py first', ha='center', va='center', transform=ax3.transAxes)
    ax3.set_title('Communication Overhead')

# Panel 4: Non-IID impact — use alpha_histories from Section 6
if alpha_histories:
    viridis = plt.cm.viridis(np.linspace(0.15, 0.85, max(len(alpha_histories), 1)))
    for i, (alpha, hist) in enumerate(sorted(alpha_histories.items())):
        rounds = [e['round'] for e in hist if 'honest_global_accuracy' in e]
        accs = [e['honest_global_accuracy'] for e in hist if 'honest_global_accuracy' in e]
        ax4.plot(rounds, accs, label=f'α={alpha}', color=viridis[i])
    ax4.set_xlabel('Round')
    ax4.set_ylabel('Accuracy')
    ax4.set_title('Non-IID Impact (Dirichlet α)')
    ax4.legend()
    ax4.set_ylim(0, 1.05)
    ax4.grid(alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'Run ablation_heterogeneity.py', ha='center', va='center', transform=ax4.transAxes)
    ax4.set_title('Non-IID Impact')

fig.suptitle('GossipRoboFL — Results Dashboard', fontsize=15, fontweight='bold')
plt.show()
fig.savefig(f'{RESULTS_DIR}/nb_dashboard.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Dashboard saved to {RESULTS_DIR}/nb_dashboard.png')

## 11. Per-Client Accuracy Distribution

Box plot showing accuracy variance across robots — measures consensus quality.

In [ ]:
# Pick one gossip experiment and plot per-client accuracy over time
gossip_keys_pc = [
    k for k in histories
    if 'fedavg' not in k and not k.startswith('noniid_') and histories[k]
]
if gossip_keys_pc:
    key = gossip_keys_pc[0]
    hist = histories[key]
    
    # Collect per-client accuracy at each eval round
    eval_entries = [e for e in hist if 'per_client_accuracy' in e and e['per_client_accuracy']]
    if eval_entries:
        rounds = [e['round'] for e in eval_entries]
        per_client = [e['per_client_accuracy'] for e in eval_entries]
        worst = [e.get('honest_worst_accuracy', min(e['per_client_accuracy'])) for e in eval_entries]
        
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        
        # Box plots at key rounds
        n_boxes = min(8, len(eval_entries))
        step = max(1, len(eval_entries) // n_boxes)
        box_data = [per_client[i] for i in range(0, len(eval_entries), step)]
        box_labels = [str(rounds[i]) for i in range(0, len(eval_entries), step)]
        
        bp = axes[0].boxplot(box_data, labels=box_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#3498db')
            patch.set_alpha(0.6)
        axes[0].set_xlabel('Round')
        axes[0].set_ylabel('Client Test Accuracy')
        axes[0].set_title(f'Per-Client Accuracy Distribution\n({key})')
        axes[0].grid(axis='y', alpha=0.3)
        axes[0].set_ylim(0, 1.05)
        
        # Std deviation over rounds (measures consensus)
        stds = [e.get('honest_accuracy_std', np.std(e['per_client_accuracy'])) for e in eval_entries]
        axes[1].plot(rounds, stds, color='#e74c3c', linewidth=2)
        axes[1].set_xlabel('Round')
        axes[1].set_ylabel('Std Dev of Client Accuracies')
        axes[1].set_title('Client Consensus (lower std = better)')
        axes[1].grid(alpha=0.3)
        axes[1].set_ylim(bottom=0)
        
        # Worst-client (honest) accuracy over rounds
        axes[2].plot(rounds, worst, color='#9b59b6', linewidth=2)
        axes[2].set_xlabel('Round')
        axes[2].set_ylabel('Worst Client Accuracy')
        axes[2].set_title('Worst-Client Accuracy (fairness)')
        axes[2].grid(alpha=0.3)
        axes[2].set_ylim(0, 1.05)
        
        plt.tight_layout()
        plt.show()
        fig.savefig(f'{RESULTS_DIR}/nb_per_client_accuracy.png', dpi=150, bbox_inches='tight')
        plt.close(fig)
    else:
        print('No per_client_accuracy data found in history.')
else:
    print('No gossip experiment found.')

## 12. Export All Plots

Regenerate all standard plots from saved history files.

In [ ]:
from src.metrics import MetricsTracker

print('Regenerating plots for all experiments...')
for name, hist in histories.items():
    if not hist:
        continue
    # Find byzantine IDs (not stored in history — use empty set for display)
    tracker = MetricsTracker(
        log_dir=RESULTS_DIR,
        run_name=f'nb_{name}',
        num_clients=len(hist[0].get('per_client_accuracy', [1])),
        byzantine_ids=set(),
    )
    tracker.history = hist
    paths = tracker.generate_plots(save_dir=RESULTS_DIR)
    tracker.close()
    if paths:
        print(f'  {name}: {len(paths)} plots')

print('Done!')